# 12주차

https://school.programmers.co.kr/learn/courses/30/lessons/214295

프로그래머스

2023 현대모비스 알고리즘 경진대회 본선

미로 주행 테스트 문제

지정 크기의 미로에서 랜덤 위치에서 제한된 이동 횟수안에 도달할 표지판 수 반환

- 입력 = [x, y, d, flag]
 - x, y: 출발점 위치
 - d: 이동 횟수
 - flag: 표지판 도착여부

In [ ]:
using System;
using System.Runtime.InteropServices;
using System.Runtime.CompilerServices;


public static class Utility
{
    public struct Rect
    {
        public int x1, y1, x2, y2;

        public Rect(int x1, int y1, int x2, int y2)
        {
            this.x1 = x1;
            this.y1 = y1;
            this.x2 = x2;
            this.y2 = y2;
        }

        public static implicit operator bool(Rect rect) => rect.x1 <= rect.x2 && rect.y1 <= rect.y2;
    }

    public static Rect clipRect(Rect rect, int x, int y, int distance, int m)
    {
        long ix = x - y + m;
        long iy = x + y;
        int leftX = (int)Math.Max(rect.x1, ix - distance);
        int topY = (int)Math.Max(rect.y1, iy - distance);
        int rightX = (int)Math.Min(rect.x2, ix + distance);
        int bottomY = (int)Math.Min(rect.y2, iy + distance);
        return new Rect(leftX, topY, rightX, bottomY);
    }

    private static int gridFitValue(int x, int y, int m) => ((x + y - m) & 1);

    private static long getOutGridCount(int x, int y, int distance, int m)
    {
        if (distance <= 0) return 0;
        int alignment = gridFitValue(x, y, m);
        distance = (distance + 1 - alignment) / 2;
        return (long)(distance + alignment) * distance;
    }

    public static long getGridCount(Rect rect, int n, int m)
    {
        int mx = m;
        int mn = n;

        if (rect.y2 < mx && rect.x1 < mx - rect.y2) rect.x1 = Math.Max(rect.x1, mx - rect.y2);
        if (rect.x2 < mx && rect.y1 < mx - rect.x2) rect.y1 = Math.Max(rect.y1, mx - rect.x2);
        if (rect.y2 < mn && rect.x2 - mx > rect.y2) rect.x2 = Math.Min(rect.x2, mx + rect.y2);
        if (rect.x1 > mx && rect.y1 < rect.x1 - mx) rect.y1 = Math.Max(rect.y1, rect.x1 - mx);
        if (rect.y1 > mx && rect.x1 < rect.y1 - mx) rect.x1 = Math.Max(rect.x1, rect.y1 - mx);
        if (rect.x2 < mn && rect.y2 - mx > rect.x2) rect.y2 = Math.Min(rect.y2, mx + rect.x2);
        if (rect.y1 > mn && rect.x2 > mn + mx - (rect.y1 - mn)) rect.x2 = Math.Min(rect.x2, mn + mx - (rect.y1 - mn));
        if (rect.x1 > mn && rect.y2 > mn + mx - (rect.x1 - mn)) rect.y2 = Math.Min(rect.y2, mn + mx - (rect.x1 - mn));

        if (!rect) return 0;

        int alignment = gridFitValue(rect.x1, rect.y1, m);
        int width = rect.x2 - rect.x1 + 1;
        int height = rect.y2 - rect.y1 + 1;

        int rightWidth = width >> 1;
        int leftWidth = width - rightWidth;

        long leftHeight = (height + 1 - alignment) >> 1;
        long rightHeight = (height + alignment) >> 1;

        long result = leftWidth * leftHeight + rightWidth * rightHeight;

        long edgeResult = 0;

        if (rect.x1 < mx - rect.y1)
            edgeResult -= getOutGridCount(rect.x1, rect.y1, mx - rect.y1 - rect.x1, mx);
        if (rect.x2 - mx > rect.y1)
            edgeResult -= getOutGridCount(rect.x2, rect.y1, rect.x2 - mx - rect.y1, mx);
        if (rect.x1 < rect.y2 - mx)
            edgeResult -= getOutGridCount(rect.x1, rect.y2, rect.y2 - mx - rect.x1, mx);
        if (rect.x2 - mn > mn + mx - rect.y2)
            edgeResult -= getOutGridCount(rect.x2, rect.y2, rect.x2 - mn - (mn + mx - rect.y2), mx);

        result += edgeResult;

        return result;
    }
}

In [ ]:
public class SegmentTree
{
    // 1. 이벤트 정보를 담는 구조체: 사각형의 x 좌표와 y 좌표 범위, 그리고 카운트 정보를 저장
    [StructLayout(LayoutKind.Sequential)]
    private struct Event : IComparable<Event>
    {
        // 4바이트 정렬을 위해 순서를 최적화 (메모리 최적화 목적)
        public int value;    // x 좌표 (이벤트가 발생하는 x 위치)
        public int count;    // 사각형의 시작(1) 혹은 끝(-1)을 나타내는 값
        public int yStart;   // 사각형의 y 시작 좌표
        public int yEnd;     // 사각형의 y 끝 좌표

        // 이벤트 간의 정렬 기준 정의 (x 좌표 우선, 그 다음 count, 그 다음 y 좌표 범위)
        [MethodImpl(MethodImplOptions.AggressiveInlining)]
        public int CompareTo(Event other)
        {
            if (value != other.value)
                return value.CompareTo(other.value);

            long diff = ((long)value << 32) | (uint)count;
            long otherDiff = ((long)other.value << 32) | (uint)other.count;

            if (diff != otherDiff)
                return (diff < otherDiff) ? -1 : 1;

            diff = ((long)yStart << 32) | (uint)yEnd;
            otherDiff = ((long)other.yStart << 32) | (uint)other.yEnd;

            return (diff < otherDiff) ? -1 : (diff > otherDiff) ? 1 : 0;
        }
    }

    // 2. 세그먼트 트리의 노드 구조체
    private struct TreeNode
    {
        public int count, link, prevLink, fill, prevFill;

        // 노드 생성자: 초기 값들을 설정 (기본값은 0)
        public TreeNode(int count = 0, int link = 0, int prevLink = 0, int fill = 0, int prevFill = 0)
        {
            this.count = count;
            this.link = link;
            this.prevLink = prevLink;
            this.fill = fill;
            this.prevFill = prevFill;
        }
    }

    // 3. 멤버 변수들
    private Event[] eventList;   // 사각형 이벤트들을 저장하는 배열
    private int[] yCoords;       // 사각형에서 사용되는 y 좌표들을 저장하는 배열
    private TreeNode[] nodes;    // 세그먼트 트리의 노드 배열
    private int eventCount, yCount, rows, cols;  // 이벤트 개수, y 좌표 개수, 그리드 행과 열

    // 4. 생성자: 트리의 용량(capacity)과 그리드 크기(n x m)를 초기화
    public SegmentTree(int capacity, int n, int m)
    {
        eventList = new Event[capacity * 2];  // 각 사각형마다 두 개의 이벤트가 있으므로 2배 크기
        yCoords = new int[capacity * 2];
        rows = n;
        cols = m;
    }

    // 5. 사각형 추가 메서드: 사각형의 좌측, 우측 경계 이벤트를 추가하고 y 좌표도 저장
    public void addRect(Utility.Rect r)
    {
        eventList[eventCount++] = new Event { value = r.x1, yStart = r.y1, yEnd = r.y2, count = 1 };
        eventList[eventCount++] = new Event { value = r.x2, yStart = r.y1, yEnd = r.y2, count = -1 };
        yCoords[yCount++] = r.y1;
        yCoords[yCount++] = r.y2;
    }

    // 6. shift 메서드: 현재 노드의 상태를 자식 노드에 전파(shift)하기 위한 메서드
    //    prevFill과 prevLink 값을 업데이트하여 이전 상태를 저장
    private void shift(int start, int end, int nodeIndex)
    {
        ref TreeNode node = ref nodes[nodeIndex];
        nodes[nodeIndex].prevFill = node.fill;
        nodes[nodeIndex].prevLink = node.link;

        // 채워진 값(fill)이 0이 아니고 4보다 작으면 자식 노드로 전파 (부분 구간 처리)
        if (node.fill != 0 && node.fill < 4)
        {
            int mid = (start + end) >> 1;
            if ((node.fill & 1) != 0) shift(start, mid, nodeIndex << 1);
            if ((node.fill & 2) != 0) shift(mid + 1, end, (nodeIndex << 1) | 1);
        }
    }

    // 7. update 메서드: y 좌표의 구간 [yStart, yEnd]에 대해 현재 이벤트의 영향을 반영
    //    노드의 count와 link 값을 업데이트하며, 완전 채워진 구간이면 fill 플래그를 설정
    private void update(int yStart, int yEnd, int start, int end, int nodeIndex, int value)
    {
        ref TreeNode node = ref nodes[nodeIndex];
        int leftChild = nodeIndex << 1, rightChild = leftChild | 1;

        // 현재 구간이 [yStart, yEnd] 구간 내에 완전히 포함되면
        if (yStart <= yCoords[start] && yCoords[end] <= yEnd)
        {
            node.count += value;
            // 구간의 끝이 yEnd 미만일 경우 link도 업데이트 (구간 연결 정보)
            if (yCoords[end] < yEnd) node.link += value;

            // count가 0이면서 내부 노드라면 자식 노드의 이전 상태를 shift 메서드를 통해 갱신
            if (node.count == 0 && start < end)
            {
                int mid = (start + end) >> 1;
                shift(start, mid, leftChild);
                shift(mid + 1, end, rightChild);
            }
        }
        // 현재 구간이 완전히 포함되지 않으면 자식 노드로 내려가서 업데이트 수행
        else if (start < end)
        {
            int mid = (start + end) >> 1;
            if (yStart <= yCoords[mid]) update(yStart, yEnd, start, mid, leftChild, value);
            if (yEnd > yCoords[mid]) update(yStart, yEnd, mid + 1, end, rightChild, value);
        }

        // 구간이 count가 0이면 fill 값을 0, 아니면 4로 설정하고 자식 노드의 fill 여부도 함께 표시
        node.fill = node.count != 0 ? 4 : 0;
        if (start < end)
        {
            if (nodes[leftChild].fill != 0) node.fill |= 1;
            if (nodes[rightChild].fill != 0) node.fill |= 2;
        }
    }

    // 8. hasPrevLink 메서드: 이전 상태(prevLink)에서 연결(링크)이 존재하는지 확인
    private bool hasPrevLink(int start, int end, int nodeIndex)
    {
        TreeNode node = nodes[nodeIndex];
        if (node.prevLink != 0) return true;
        if ((node.prevFill & 2) != 0)
        {
            int mid = (start + end) >> 1;
            start = mid + 1;
            nodeIndex = (nodeIndex << 1) | 1;
            // 현재 노드가 채워진 상태이면 현재 링크 여부를, 아니라면 이전 링크 여부를 재귀적으로 체크
            return (node.fill & 4) != 0 ? hasLink(start, end, nodeIndex) : hasPrevLink(start, end, nodeIndex);
        }
        return false;
    }

    // 9. hasLink 메서드: 현재 상태(fill)에서 연결(link)이 존재하는지 확인
    private bool hasLink(int start, int end, int nodeIndex)
    {
        TreeNode node = nodes[nodeIndex];
        if (node.link != 0) return true;
        if ((node.fill & 2) != 0)
        {
            // 오른쪽 자식 노드로 이동하면서 링크가 존재하는지 재귀적으로 확인
            start = (start + end) >> 1 + 1;
            nodeIndex = (nodeIndex << 1) | 1;
            return hasLink(start, end, nodeIndex);
        }
        return false;
    }

    // 10. getPrevSum 메서드: 이전 상태(prevFill, prevLink)를 바탕으로 이전 구간의 합을 구함
    //     Utility.getGridCount를 호출하여 그리드 상의 누적 개수를 계산
    private long getPrevSum(int start, int end, int nodeIndex, int x, bool prev)
    {
        TreeNode node = nodes[nodeIndex];
        if ((node.fill & 4) != 0) prev = false;
        int fill = prev ? node.prevFill : node.fill;

        if ((fill & 4) != 0)
        {
            int yStart = yCoords[start];
            int yEnd = end < yCount - 1 && hasPrevLink(start, end, nodeIndex) ? yCoords[end + 1] - 1 : yCoords[end];
            return Utility.getGridCount(new Utility.Rect(x, yStart, x, yEnd), rows, cols);
        }
        else if ((fill & 3) != 0)
        {
            long result = 0;
            int mid = (start + end) >> 1;
            if ((fill & 1) != 0) result += getPrevSum(start, mid, nodeIndex << 1, x, prev);
            if ((fill & 2) != 0) result += getPrevSum(mid + 1, end, (nodeIndex << 1) | 1, x, prev);
            return result;
        }
        return 0;
    }

    // 11. calculateSum 메서드: 현재 구간의 합을 계산하는 메서드로, 채워진 구간과 링크 정보를 활용하여
    //     Utility.getGridCount로 실제 그리드 상의 개수를 구한 후, 이전 상태의 값과 차감하는 방식으로 합산
    private long calculateSum(int start, int end, int nodeIndex, int x1, int x2, int prevFill)
    {
        ref TreeNode node = ref nodes[nodeIndex];
        if (prevFill == 0 && (node.prevFill & 4) != 0) prevFill = 1;

        long result = 0;

        if ((node.fill & 4) != 0)
        {
            bool linked = end < yCount - 1 && hasLink(start, end, nodeIndex);
            int yStart, yEnd;

            if (prevFill == 1)
            {
                if (linked && !hasPrevLink(start, end, nodeIndex))
                {
                    yStart = yCoords[end] + 1;
                    yEnd = yCoords[end + 1] - 1;
                    if (yStart <= yEnd)
                    {
                        result += Utility.getGridCount(new Utility.Rect(x1, yStart, x1, yEnd), rows, cols);
                    }
                }

                if (x1 == x2)
                {
                    node.prevFill = node.fill;
                    node.prevLink = node.link;
                    return result;
                }
                x1++;
            }

            yStart = yCoords[start];
            yEnd = linked ? yCoords[end + 1] - 1 : yCoords[end];
            result += Utility.getGridCount(new Utility.Rect(x1, yStart, x2, yEnd), rows, cols);

            if (prevFill == 0 && (node.prevFill & 3) != 0)
            {
                int mid = (start + end) >> 1;
                if ((node.prevFill & 1) != 0) result -= getPrevSum(start, mid, nodeIndex << 1, x1, true);
                if ((node.prevFill & 2) != 0) result -= getPrevSum(mid + 1, end, (nodeIndex << 1) | 1, x1, true);
            }
        }
        else if ((node.fill & 3) != 0)
        {
            int mid = (start + end) >> 1;
            if ((node.fill & 1) != 0) result += calculateSum(start, mid, nodeIndex << 1, x1, x2, (node.prevFill & 1) != 0 ? prevFill : -1);
            if ((node.fill & 2) != 0) result += calculateSum(mid + 1, end, (nodeIndex << 1) | 1, x1, x2, (node.prevFill & 2) != 0 ? prevFill : -1);
        }

        // 현재 노드의 이전 상태를 현재 상태로 갱신
        node.prevFill = node.fill;
        node.prevLink = node.link;

        return result;
    }

    // 12. getTotalSum 메서드: 전체 이벤트들을 처리하여 최종 결과(누적 합)를 계산하는 엔트리 포인트
    public long getTotalSum()
    {
        if (yCount == 0) return 0;

        // 저장된 y 좌표들을 정렬하여 중복 제거
        Array.Sort(yCoords, 0, yCount);
        int uniqueYCount = 1;
        for (int i = 1; i < yCount; ++i)
        {
            if (yCoords[i] != yCoords[i - 1]) yCoords[uniqueYCount++] = yCoords[i];
        }
        yCount = uniqueYCount;

        // 이벤트들을 x 좌표 기준으로 정렬
        Array.Sort(eventList, 0, eventCount);

        // 세그먼트 트리의 노드 개수를 y 좌표 개수에 맞게 결정 (완전 이진 트리 형태)
        int nodeCount = 1;
        while (nodeCount < yCount) nodeCount <<= 1;
        nodes = new TreeNode[nodeCount * 2];

        long result = 0;
        int yEnd = yCount - 1;

        // 스윕 라인 알고리즘: 이벤트들을 순차적으로 처리하며, 각 이벤트 사이의 x 구간에 대해 합산 계산
        for (int i = 0; i < eventCount; ++i)
        {
            Event e = eventList[i];
            if (i > 0)
            {
                result += calculateSum(0, yEnd, 1, eventList[i - 1].value, e.value, 0);
            }
            update(e.yStart, e.yEnd, 0, yEnd, 1, e.count);
        }

        return result;
    }
}


In [ ]:
public class Solution
{
    public long solution(int n, int m, int[,] tests)
    {
        return Solve(n, m, tests);
    }

    public long Solve(int n, int m, int[,] tests)
    {
        Utility.Rect region = new Utility.Rect(0, 0, n + m, n + m);
        int failureCount = 0;

        int sizeTests = tests.GetLength(0);
        for (int i = 0; i < sizeTests; ++i)
        {
            int x = tests[i, 0];
            int y = tests[i, 1];
            bool isValid = tests[i, 3] != 0;

            if (isValid)
            {
                int distance = tests[i, 2];
                region = Utility.clipRect(region, x, y, distance, m);
                if (!region) return 0;
            }
            else
            {
                failureCount++;
            }
        }

        long result = Utility.getGridCount(region, n, m);
        if (result <= 0) return 0;

        if (failureCount > 0)
        {
            SegmentTree segmentTree = new SegmentTree(failureCount, n, m);

            for (int i = 0; i < sizeTests; ++i)
            {
                int x = tests[i, 0];
                int y = tests[i, 1];
                bool isValid = tests[i, 3] != 0;

                if (!isValid)
                {
                    int distance = tests[i, 2];
                    Utility.Rect subRegion = Utility.clipRect(region, x, y, distance, m);
                    if (subRegion)
                    {
                        segmentTree.addRect(subRegion);
                    }
                }
            }

            result -= segmentTree.getTotalSum();
        }

        return result;
    }
}

## python 변환

In [ ]:
class Utility:
  def Rect(self, x1, y1, x2, y2):
    self.x1 = x1
    self.y1 = y1
    self.x2 = x2
    self.y2 = y2

    return (self.x1, self.y1, self.x2, self.y2)

  def clipRect(self, x, y, distance, m):
    ix = x - y + m
    iy = x + y
    left = max(self.x1, ix - distance)
    top = max(self.y1, iy - distance)
    right = min(self.x2, ix + distance)
    bottom = min(self.y2, iy + distance)

    return (left, top, right, bottom)

  def gridFitValue(x, y, m):
    return ((x + y - m) & 1)

  def getOutGridCount(x, y, distance, m):
    if (distance <= 0):
      return 0
    alignment = Utility.gridFitValue(x, y, m)
    distance = int((distance + 1 - alignment) / 2)

    return (distance + alignment) * distance

  def getGridCount(self, n, m):
    mx = m
    mn = n

    if self.y2 < mx and self.x1 < mx - self.y2:
      self.x1 = max(self.x1, mx - self.y2)
    if self.x2 < mx and self.y1 < mx - self.x2:
      self.y1 = max(self.y1, mx - self.x2)
    if self.y2 < mn and self.x2 - mx > self.y2:
      self.x2 = min(self.x2, mx + self.y2)
    if self.x1 > mx and self.y1 < self.x1 - mx:
      self.y1 = max(self.y1, self.x1 - mx)
    if self.y1 > mx and self.x1 < self.y1 - mx:
      self.x1 = max(self.x1, self.y1 - mx)
    if self.x2 < mn and self.y2 - mx > self.x2:
      self.y2 = min(self.y2, mx + self.x2)
    if self.y1 > mn and self.x2 > mn + mx - (self.y1 - mn):
      self.x2 = min(self.x2, mn + mx - (self.y1 - mn))
    if self.x1 > mn and self.y2 > mn + mx - (self.x1 - mn):
      self.y2 = min(self.y2, mn + mx - (self.x1 - mn))

    # if (!rect) return 0;

    alignment = Utility.gridFitValue(self.x1, self.y1, m)
    width = self.x2 - self.x1 + 1
    height = self.y2 - self.y1 + 1

    rightWidth = width >> 1
    leftWidth = width - rightWidth

    leftHeight = (height + 1 - alignment) >> 1
    rightHeight = (height + alignment) >> 1

    result = leftWidth * leftHeight + rightWidth * rightHeight

    # 경계를 넘은 node 제거
    edgeResult = 0

    if self.x1 < mx - self.y1:
      edgeResult -= Utility.getOutGridCount(self.x1, self.y1, mx - self.y1 - self.x1, mx)
    if self.x2 - mx > self.y1:
      edgeResult -= Utility.getOutGridCount(self.x2, self.y1, self.x2 - mx - self.y1, mx)
    if self.x1 < self.y2 - mx:
      edgeResult -= Utility.getOutGridCount(self.x1, self.y2, self.y2 - mx - self.x1, mx)
    if self.x2 - mn > mn + mx - self.y2:
      edgeResult -= Utility.getOutGridCount(self.x2, self.y2, self.x2 - mn - (mn + mx - self.y2), mx)

    result += edgeResult;
    print(edgeResult)

    return result

In [ ]:
class Event:
  def __init__(self, value, count, yStart, yEnd):
    self.value = value
    self.count = count
    self.yStart = yStart
    self.yEnd = yEnd

class SegmentTree:
  # 생성자: 트리의 용량(capacity)과 그리드 크기(n x m)를 초기화
  def __init__(self, capacity, n, m):
    self.eventList = []
    self.yCoords = []
    self.rows = n
    self.cols = m

  # 사각형 추가 메서드: 사각형의 좌측, 우측 경계 이벤트를 추가하고 y 좌표도 저장
  def addRect(self, r):
    self.eventList.append(Event(value = r[0], yStart = r[1], yEnd = r[3], count = 1))
    self.eventList.append(Event(value = r[2], yStart = r[1], yEnd = r[3], count = -1))
    self.yCoords.append(r[1])
    self.yCoords.append(r[3])

  # shift 메서드: 현재 노드의 상태를 자식 노드에 전파(shift)하기 위한 메서드
  # prevFill과 prevLink 값을 업데이트하여 이전 상태를 저장
  def shift(start, end, nodeIndex):
    ref TreeNode node = ref nodes[nodeIndex]
    nodes[nodeIndex].prevFill = node.fill
    nodes[nodeIndex].prevLink = node.link

    # 채워진 값(fill)이 0이 아니고 4보다 작으면 자식 노드로 전파 (부분 구간 처리)
    if node.fill != 0 and node.fill < 4:
      mid = int((start + end) >> 1)
      if ((node.fill & 1) != 0):
        shift(start, mid, nodeIndex << 1)
      if ((node.fill & 2) != 0):
        shift(mid + 1, end, (nodeIndex << 1) | 1)

In [ ]:
def solution(n, m, tests):
  a = Utility()
  region = a.Rect(0, 0, n + m, n + m)
  failureCount = 0

  sizeTests = len(tests)
  for i in range(sizeTests):
    x = tests[i][0]
    y = tests[i][1]
    isValid = tests[i][3] != 0

    if isValid:
      distance = tests[i][2]
      region = a.clipRect(x, y, distance, m)
      region = a.Rect(*region)

      if not region:
        return 0

    else:
      failureCount += 1

  result = a.getGridCount(n, m)

  if (result <= 0) return 0;

  if failureCount > 0:
    segmentTree = SegmentTree(failureCount, n, m)
    for i in range(sizeTests):
      x = tests[i][0]
      y = tests[i][1]
      isValid = tests[i][3] != 0

      if not isValid:
        distance = tests[i][2]
        subRegion = a.clipRect(x, y, distance, m)
          if subRegion:
            segmentTree.addRect(subRegion)

      result -= segmentTree.getTotalSum()

  return result

In [ ]:
n, m, tests	= 3, 5, [[2, 3, 2, 1], [1, 0, 4, 0], [0, 4, 1, 0]]

solution(n, m, tests)

1
-1


12